# This is an example of how to get Landsat images with Google Earth Engine

**To-Do:**
1. Register for Google Earth Engine (Free Noncommercial Tier)
enable the Service Usage Consumer and  Earth Engine Resource Writer roles: https://earthengine.google.com/signup
2. create a service account: https://console.cloud.google.com/iam-admin/serviceaccounts*
3. Assign roles: Earth Engine Resource Writer and Service usage consumer
4. get the JSON Key

In [43]:
!pip install earthengine-api

Import neccesary libraries

In [44]:
import ee
import json
import folium

**Here we use the Google Earth Engine JSON validation, not the API key!**

Add it to the notebook with Secrets.

In [45]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["GOOGLE_MAPS_API_KEY"] = "" 
secrets = UserSecretsClient()
key_json = secrets.get_secret("GoogleEarthEngineJSON")
key_data = json.loads(key_json)

credentials = ee.ServiceAccountCredentials(
    email=key_data["client_email"],
    key_data=key_json
)
ee.Initialize(credentials, project=key_data["project_id"])
print("✅ GEE OK:", key_data["project_id"])

✅ GEE OK: ancient-medium-432114-u6


Get the [LANDSAT/LC08/C02/T1_L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2#description) image from 2023 about Budapest.

In [48]:
# Budapest Landsat 8
budapest = ee.Geometry.Point([19.0402, 47.4979])

landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(budapest)
    .filterDate('2023-06-01', '2023-09-30')
    .filter(ee.Filter.lt('CLOUD_COVER', 10))
    .sort('CLOUD_COVER')
    .first()
)

vis_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 7000,
    'max': 20000,
    'gamma': 1.4
}

# Access to GEE tile URL
map_id = landsat.getMapId(vis_params)
tile_url = map_id['tile_fetcher'].url_format

# Folium map
m = folium.Map(location=[47.4979, 19.0402], zoom_start=10)

folium.TileLayer(
    tiles=tile_url,
    attr='Google Earth Engine',
    name='Landsat 8 - Budapest',
    overlay=True,
    control=True
).add_to(m)

folium.LayerControl().add_to(m)
m